**Lendo os arquivos CSV do MinIO e gravando no formato Delta**

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

load_dotenv(find_dotenv(), override=True)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

builder = SparkSession.builder \
    .appName("LandingToBronze") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.connection.request.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000") \
    .config(
    "spark.hadoop.fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

spark = configure_spark_with_delta_pip(
    builder, 
    extra_packages=["org.apache.hadoop:hadoop-aws:3.4.2"]
).getOrCreate()

df_sales = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://landing-zone/games.csv")

df_sales.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://bronze/games")

print("Tabela games convertida para Delta Lake com sucesso!")

:: loading settings :: url = jar:file:/home/jeanvv/.cache/pypoetry/virtualenvs/apache-spark-minio-z9HCMJ8B-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jeanvv/.ivy2.5.2/cache
The jars for the packages stored in: /home/jeanvv/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b3c6a66b-902f-4ff2-aea8-fa3b30495abc;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in local-m2-cache
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.2 in local-m2-cache
	found io.unitycatalog#un

Tabela games convertida para Delta Lake com sucesso!


In [2]:
df_bronze = spark.read.format("delta").load("s3a://bronze/games")

df_bronze.printSchema()
df_bronze.show(5, truncate=False)

print(f"Total de registros: {df_bronze.count()}")

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: integer (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)



+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|id |name           |platform         |release_date|metascore|user_score|developer              |publisher                               |genre                |
+---+---------------+-----------------+------------+---------+----------+-----------------------+----------------------------------------+---------------------+
|1  |#BLUD          |PC               |2024-06-18  |73       |5.6       |Exit 73 Studios        |['Humble Games']                        |Action Adventure     |
|2  |#DRIVE         |iOS (iPhone/iPad)|2021-02-16  |69       |6.8       |Dariusz Pietrala       |['PM Studios Inc.', 'Limited Run Games']|Auto Racing          |
|3  |#IDARB         |Xbox One         |2015-01-30  |77       |6.9       |Other Ocean Interactive|['Other Ocean Interactive']             |Party                |
|4  |#KILLALLZOMBIES|PC           

[Stage 16:==================================================>     (45 + 4) / 50]

Total de registros: 21910


In [4]:
load_dotenv(find_dotenv(), override=True)

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY"),
)

In [5]:
response = s3.list_objects_v2(Bucket="landing-zone")

arquivos = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".csv")
]

for arquivo in arquivos:
    nome_tabela = arquivo.rsplit("/", 1)[-1][:-4]

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"s3a://landing-zone/{arquivo}")

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"s3a://bronze/{nome_tabela}")

    print(f"{nome_tabela}: {df.count()} registros convertidos")

agendamento: 10 registros convertidos


cliente: 10 registros convertidos


funcionario: 10 registros convertidos


games: 21910 registros convertidos


historico: 10 registros convertidos


how_long_to_beat: 13196 registros convertidos


pet: 10 registros convertidos


regional_sales: 16608 registros convertidos


servico: 10 registros convertidos


steam_metrics: 3178 registros convertidos
walkthrough_links: 0 registros convertidos


In [6]:
response = s3.list_objects_v2(Bucket="bronze")

for obj in response.get("Contents", []):
    print(obj["Key"])

agendamento/_delta_log/00000000000000000000.crc
agendamento/_delta_log/00000000000000000000.json
agendamento/_delta_log/_staged_commits/
agendamento/part-00000-18c98dae-47d6-4a6e-ab3d-693daec234b9-c000.snappy.parquet
cliente/_delta_log/00000000000000000000.crc
cliente/_delta_log/00000000000000000000.json
cliente/_delta_log/_staged_commits/
cliente/part-00000-cb72aaf5-4c7a-4bbe-8d89-1a52d5b9b56f-c000.snappy.parquet
funcionario/_delta_log/00000000000000000000.crc
funcionario/_delta_log/00000000000000000000.json
funcionario/_delta_log/_staged_commits/
funcionario/part-00000-e7699dca-3162-4bf4-bdda-d76357724fc7-c000.snappy.parquet
games/_delta_log/00000000000000000000.crc
games/_delta_log/00000000000000000000.json
games/_delta_log/00000000000000000001.crc
games/_delta_log/00000000000000000001.json
games/_delta_log/_staged_commits/
games/part-00000-47c51722-1988-49a9-aca8-f4541833d501-c000.snappy.parquet
games/part-00000-efb1c70d-5675-4bca-98c7-65ecc4224d74-c000.snappy.parquet
historico/_de